# Asistente Bancario — Inferencia y Evaluación

## GenAI Lifecycle: Evaluación

**Objetivo:** Comparar el modelo base (Gemma 3 4B) vs el modelo fine-tuneado para determinar si el fine-tuning mejoró las respuestas en el dominio bancario.

**Método:** Evaluación cualitativa (comparación de respuestas) + cuantitativa (similitud con respuestas esperadas del dataset de test).

**Nota:** El resultado puede ser positivo o negativo. Ambos son válidos como conclusión del proyecto.

In [1]:
!pip install -qU unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.7/405.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import unsloth
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from huggingface_hub import notebook_login
import torch

notebook_login()

In [5]:
HF_USERNAME = "testlegadoss"
model, tokenizer = FastModel.from_pretrained(
    model_name=f"{HF_USERNAME}/gemma3-4b-banking-assistant-es",
    max_seq_length=2048,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

def generate_response(question, max_tokens=200):
    messages = [{"role": "user", "content": [{"type": "text", "text": question}]}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(text=text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7, top_p=0.9)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("model\n")[-1]

==((====))==  Unsloth 2026.1.4: Fast Gemma3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/59.7M [00:00<?, ?B/s]

## 2. Preguntas de evaluación

Armo 8 preguntas que cubren distintas categorías bancarias: apertura de cuentas, transferencias, tarjetas, préstamos, reclamos, y banca digital. La idea es que sean preguntas realistas que haría un cliente, no preguntas artificiales.

In [6]:
test_questions = [
    "¿Cómo puedo abrir una cuenta de ahorro?",
    "¿Cuál es el límite de transferencia diario?",
    "Necesito bloquear mi tarjeta de crédito",
    "¿Qué comisiones tiene la cuenta corriente?",
    "¿Cómo solicito un préstamo personal?",
    "Quiero hacer un reclamo por un cobro no reconocido",
    "¿Cuáles son los requisitos para abrir una cuenta?",
    "¿Tienen servicio de banca móvil?",
]

print("=== Respuestas del modelo FINE-TUNEADO ===\n")
ft_responses = []
for q in test_questions:
    resp = generate_response(q)
    ft_responses.append(resp)
    print(f"Q: {q}")
    print(f"A: {resp}\n")

=== Respuestas del modelo FINE-TUNEADO ===

Q: ¿Cómo puedo abrir una cuenta de ahorro?
A: Estoy aquí para ayudarte a abrir una cuenta de ahorro. Abre una cuenta de ahorro para guardar tu dinero y hacer que crezca. Aquí hay algunos pasos que puedes seguir: 1. Visita nuestro sitio web de banca en línea o abre nuestra aplicación bancaria. 2. Busca la pestaña "Cuentas" o "Cuenta de ahorro". 3. Haz clic en ella y selecciona "Open a Savings Account". 4. Rellena los datos requeridos, tales como tu nombre, dirección y información de contacto. 5. Selecciona la cantidad de apertura que deseas abrir. 6. Revise los términos y condiciones y acepta cualquier acuerdo requerido. 7. Si es necesario, provee cualquier documentación o documentos adicionales que se te pida. 8. Revisa la información para asegurarte de que es exacto. 10. Haz clic en "Siguiente" o "Enviar" para confirmar tu solicitud. 11. Una vez que tu solicitud

Q: ¿Cuál es el límite de transferencia diario?
A: Puedo ayudarte con eso! El lí

## 3. Comparación con el modelo base

Ahora cargo el mismo modelo pero sin fine-tuning, para ver si realmente hubo mejora. Si el modelo base ya responde bien, significaría que el fine-tuning no aportó mucho — y eso también es un resultado válido para el proyecto.

In [7]:
# Cargar modelo base (sin fine-tuning) para comparar
model_base, tokenizer_base = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it",
    max_seq_length=2048,
    load_in_4bit=True,
)
tokenizer_base = get_chat_template(tokenizer_base, chat_template="gemma-3")

def generate_base(question, max_tokens=200):
    messages = [{"role": "user", "content": [{"type": "text", "text": question}]}]
    text = tokenizer_base.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer_base(text=text, return_tensors="pt").to("cuda")
    outputs = model_base.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7, top_p=0.9)
    return tokenizer_base.decode(outputs[0], skip_special_tokens=True).split("model\n")[-1]

print("=== Respuestas del modelo BASE ===\n")
base_responses = []
for q in test_questions:
    resp = generate_base(q)
    base_responses.append(resp)
    print(f"Q: {q}")
    print(f"A: {resp}\n")

==((====))==  Unsloth 2026.1.4: Fast Gemma3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
=== Respuestas del modelo BASE ===

Q: ¿Cómo puedo abrir una cuenta de ahorro?
A: Abrir una cuenta de ahorro es un proceso relativamente sencillo, aunque los detalles pueden variar ligeramente dependiendo de la institución financiera. Aquí te explico los pasos generales y las opciones disponibles:

**1. Investigación y Elección de la Institución Financiera:**

* **Bancos Tradicionales:** Ofrecen una amplia ga

In [8]:
import pandas as pd

comparison = pd.DataFrame({
    'Pregunta': test_questions,
    'Modelo Base': [r[:150] + '...' for r in base_responses],
    'Fine-tuneado': [r[:150] + '...' for r in ft_responses],
})
comparison

,Pregunta,Modelo Base,Fine-tuneado
0,¿Cómo puedo abrir una cuenta de ahorro?,Abrir una cuenta de ahorro es un proceso relat...,Estoy aquí para ayudarte a abrir una cuenta de...
1,¿Cuál es el límite de transferencia diario?,El límite de transferencia diaria de dinero va...,Puedo ayudarte con eso! El límite de transfere...
2,Necesito bloquear mi tarjeta de crédito,Bloquear tu tarjeta de crédito es una medida i...,Nuestro equipo está aquí para ayudarle a bloqu...
3,¿Qué comisiones tiene la cuenta corriente?,Las comisiones que puede tener una cuenta corr...,Estoy aquí para ayudarte con los cargos asocia...
4,¿Cómo solicito un préstamo personal?,Solicitar un préstamo personal puede parecer u...,Estoy aquí para ayudarte a solicitar un présta...
5,Quiero hacer un reclamo por un cobro no recono...,¡Claro! Te puedo ayudar a redactar un reclamo ...,Estoy aquí para ayudarle con su reclamación. E...
6,¿Cuáles son los requisitos para abrir una cuenta?,Los requisitos para abrir una cuenta bancaria ...,Estoy aquí para ayudarle con eso. Abrir una cu...
7,¿Tienen servicio de banca móvil?,"Sí, tenemos servicio de banca móvil. Puedes ac...",Nuestro banco ofrece un servicio de banca móvi...


### Conclusiones — Fase Evaluación

### Observaciones cualitativas

Al comparar las respuestas de ambos modelos sobre las 8 preguntas de evaluación, se observa una diferencia clara en el tono y el enfoque. El modelo fine-tuneado adopta consistentemente el rol de un asistente bancario, utilizando expresiones como "Estoy aquí para ayudarte" o "Nuestro equipo está aquí para ayudarle", mientras que el modelo base responde de forma genérica e informativa, sin asumir un rol institucional. Además, las respuestas del modelo fine-tuneado están orientadas a la acción y al servicio al cliente, guiando al usuario paso a paso, a diferencia del modelo base que tiende a dar explicaciones generales similares a las de un artículo informativo. También se destaca que el fine-tuning logró un estilo uniforme y un registro más formal y profesional a lo largo de todas las preguntas, lo cual es clave para una experiencia de usuario coherente en un chatbot bancario.

### ¿Mejoró el fine-tuning?

Sí. Si bien el modelo base (Gemma 3 4B Instruct) ya es capaz de responder preguntas bancarias con información correcta, el fine-tuning aportó mejoras claras en tres aspectos: el modelo fine-tuneado se comporta como un agente de atención al cliente y no como un asistente genérico; las respuestas guían al usuario hacia los próximos pasos concretos en lugar de solo informar; y el tono es profesional y consistente, lo cual es un requisito fundamental para un despliegue real en una entidad bancaria. En definitiva, la mejora no es tanto en la precisión factual —ambos modelos dan información razonable— sino en la experiencia de usuario y la adecuación al dominio.

### Limitaciones

La comparación se realizó de forma manual sobre 8 preguntas, por lo que una evaluación más robusta requeriría métricas automáticas (BLEU, ROUGE, BERTScore) y un conjunto de test más amplio. Al no estar conectado a una base de datos real, el modelo fine-tuneado puede inventar detalles como montos, plazos o requisitos que no corresponden a un banco específico; esta limitación se aborda en el Notebook 4 mediante RAG. Por último, esta evaluación